In [ ]:
from datetime import date

def calculate_use_priority(days_remaining):
    """Rank reminder urgency; this does not determine food safety."""
    if days_remaining is None:
        return "UNKNOWN"
    if days_remaining <= 2:
        return "HIGH"
    if days_remaining <= 5:
        return "MEDIUM"
    return "LOW"


# First check: no storage guidance is being estimated yet.
for days in [None, -1, 0, 2, 3, 5, 6]:
    print(f"Days remaining: {str(days):>4} → {calculate_use_priority(days)}")

In [ ]:
def evaluate_reminder(item_id, reminder_date=None, *, as_of):
    """Use a supplied reminder date without estimating storage duration."""
    days_remaining = (
        (date.fromisoformat(reminder_date) - as_of).days
        if reminder_date is not None else None
    )

    if days_remaining is None:
        status = "MISSING_DATE"
    elif days_remaining < 0:
        status = "PAST_REMINDER"
    elif days_remaining == 0:
        status = "TODAY"
    else:
        status = "UPCOMING"

    return {
        "item_id": item_id,
        "reminder_date": reminder_date,
        "use_priority": calculate_use_priority(days_remaining),
        "days_remaining": days_remaining,
        "reminder_status": status,
    }


# Fixed dates make this example reproducible.
print(evaluate_reminder(
    "demo-001", "2026-10-15", as_of=date(2026, 10, 13)
))
print(evaluate_reminder(
    "demo-002", as_of=date(2026, 10, 13)
))

In [ ]:
expected_priorities = {
    None: "UNKNOWN",
    -1: "HIGH",
    0: "HIGH",
    2: "HIGH",
    3: "MEDIUM",
    5: "MEDIUM",
    6: "LOW",
}

for days, expected in expected_priorities.items():
    assert calculate_use_priority(days) == expected, (
        f"Priority failed for {days} days"
    )

reference_date = date(2026, 10, 13)

date_cases = [
    ("2026-10-12", -1, "PAST_REMINDER"),
    ("2026-10-13", 0, "TODAY"),
    ("2026-10-15", 2, "UPCOMING"),
    (None, None, "MISSING_DATE"),
]

for reminder, expected_days, expected_status in date_cases:
    result = evaluate_reminder(
        "test-item", reminder, as_of=reference_date
    )
    assert result["days_remaining"] == expected_days
    assert result["reminder_status"] == expected_status
    assert result["reminder_date"] == reminder

print("Passed: 7 priority cases and 4 reminder-date cases.")


In [ ]:
from pathlib import Path
import csv
csv_path = Path("data/storage_rules_clean.csv")
if not csv_path.exists(): csv_path = Path("../data/storage_rules_clean.csv")
with csv_path.open(newline="", encoding="utf-8-sig") as f:
    storage_rules = list(csv.DictReader(f))
supported_rule = next(row for row in storage_rules if row["source_record_id"] == "FS-COLD-020")
print("Records loaded:", len(storage_rules))


In [ ]:
supported_rule = next(
    row for row in storage_rules
    if row["source_record_id"] == "FS-COLD-020"
)

for field in [
    "source_record_id",
    "food_group",
    "food_type",
    "storage_location",
    "refrigerator_min_days",
    "refrigerator_max_days",
    "duration_condition",
    "source_url",
]:
    print(f"{field}: {supported_rule[field]}")


## First supported storage rule
- Rule ID: FS-COLD-020
- Food: cooked meat or poultry leftovers
- Storage: refrigerator
- Required date: prepared date
- Dataset guidance: 3–4 days
- Prototype choice: use day 3 as the estimated use-soon reminder.
- Preserve the prepared date and retain the full source range.
- Missing context or unsupported food/storage returns UNKNOWN.

This is a reminder calculation, not a determination of food safety.
The day-3 choice is a prototype policy pending team review.

In [ ]:
supported_rule = next(
    row for row in storage_rules
    if row["source_record_id"] == "FS-COLD-020"
)

print("Rule:", supported_rule["source_record_id"])
print("Minimum days:", supported_rule["refrigerator_min_days"])
print("Maximum days:", supported_rule["refrigerator_max_days"])

In [ ]:
from datetime import timedelta

def evaluate_leftovers(item, *, as_of):
    """Apply FS-COLD-020 only to explicitly confirmed matching items."""
    result = {
        "item_id": item["item_id"],
        "use_priority": "UNKNOWN",
        "days_remaining": None,
        "recommended_use_date": None,
        "prepared_date": item.get("prepared_date"),
        "reminder_status": "UNKNOWN",
        "source": None,
        "reason": "",
    }

    if (
        item.get("food_type") != "Cooked meat or poultry"
        or item.get("storage_location") != "REFRIGERATOR"
    ):
        result["reason"] = "Food or storage is unsupported by this rule."
        return result

    if not item.get("prepared_date"):
        result["reason"] = "Prepared date is required."
        return result

    try:
        prepared = date.fromisoformat(item["prepared_date"])
    except (TypeError, ValueError):
        result["reason"] = "Prepared date must be a valid YYYY-MM-DD date."
        return result

    if prepared > as_of:
        result["reason"] = "Prepared date is in the future; confirm it."
        return result

    reminder = prepared + timedelta(
        days=int(supported_rule["refrigerator_min_days"])
    )
    countdown = evaluate_reminder(
        item["item_id"], reminder.isoformat(), as_of=as_of
    )

    result.update({
        "use_priority": countdown["use_priority"],
        "days_remaining": countdown["days_remaining"],
        "recommended_use_date": reminder.isoformat(),
        "reminder_status": countdown["reminder_status"],
        "source": supported_rule["source_url"],
        "reason": "Estimated day-3 reminder using FS-COLD-020.",
    })
    return result


demo_item = {
    "item_id": "leftovers-001",
    "food_type": "Cooked meat or poultry",
    "storage_location": "REFRIGERATOR",
    "prepared_date": "2026-10-12",
}

print(evaluate_leftovers(demo_item, as_of=date(2026, 10, 13)))

In [ ]:
reference_date = date(2026, 10, 13)

cases = [
    ("Valid leftovers", {}, "HIGH", 2),
    ("Missing prepared date", {"prepared_date": None}, "UNKNOWN", None),
    ("Invalid date", {"prepared_date": "not-a-date"}, "UNKNOWN", None),
    ("Future date", {"prepared_date": "2026-10-14"}, "UNKNOWN", None),
    ("Unsupported storage", {"storage_location": "FREEZER"}, "UNKNOWN", None),
    ("Unsupported food", {"food_type": "Spinach"}, "UNKNOWN", None),
    ("Reminder today", {"prepared_date": "2026-10-10"}, "HIGH", 0),
    ("Past reminder", {"prepared_date": "2026-10-09"}, "HIGH", -1),
]

for label, changes, expected_priority, expected_days in cases:
    item = {**demo_item, **changes}
    original = item.copy()
    result = evaluate_leftovers(item, as_of=reference_date)

    assert result["use_priority"] == expected_priority, label
    assert result["days_remaining"] == expected_days, label
    assert item == original, f"{label}: input was changed"
    assert result["prepared_date"] == item.get("prepared_date"), label

    print(f"PASS: {label}")

print(f"\nPassed: {len(cases)} leftovers cases.")


## Evaluation results — initial prototype
Reference date: 2026-10-13

- Priority cases: 7/7 passed.
- Reminder-date cases: 4/4 passed.
- Leftovers cases: 8/8 passed.
- Total: 19/19 passed.

Checks covered priority boundaries, date countdowns, missing/invalid/future
prepared dates, unsupported food/storage, and preservation of inputs.

Limitations:
- Automatic storage guidance currently supports only refrigerated
  cooked meat or poultry leftovers.
- Non-meat guidance, recall integration, Role 5 ranking integration,
  and user-facing notifications are not yet implemented or tested.
- Passing these tests does not establish food safety.

## Pending team decision: non-meat demo foods
The team has not yet confirmed the demo foods.

Role 3 needs the exact food types, storage locations, and relevant
conditions (such as opened/unopened or whole/cut) before selecting
and verifying additional storage rules.

In [ ]:
import json

sample_items = [
    {**demo_item, "item_id": "sample-001"},
    {
        **demo_item,
        "item_id": "sample-002",
        "prepared_date": None,
    },
    {
        **demo_item,
        "item_id": "sample-003",
        "prepared_date": "2026-10-09",
    },
]

sample_outputs = [
    evaluate_leftovers(item, as_of=date(2026, 10, 13))
    for item in sample_items
]

print(json.dumps(sample_outputs, indent=2))


In [ ]:
print(repr(supported_rule["source_url"]))

In [ ]:
from urllib.parse import urlsplit
def normalize_source_url(value):
    url = value.strip()
    parts = urlsplit(url)
    if parts.scheme != 'https' or not parts.netloc: raise ValueError('Invalid URL')
    return url
clean_url = normalize_source_url(supported_rule["source_url"])
assert clean_url == "https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts"
print("PASS: source is a plain URL.")

## Role 3 Handoff to Samira — Initial Prototype

Owner: Mahdi Mohammadkhani
Status: Working prototype; not yet integration-ready.

### Implemented
- Loads the cleaned 20-record storage dataset.
- Applies rule FS-COLD-020 to refrigerated cooked meat/poultry leftovers.
- Calculates an estimated day-3 reminder from the prepared date.
- Preserves the original prepared date.
- Returns HIGH, MEDIUM, LOW, or UNKNOWN priority.
- Includes 19 passing calculation/test cases and 3 sample outputs.

### Inputs for evaluate_leftovers()
- item_id
- food_type: "Cooked meat or poultry"
- storage_location: "REFRIGERATOR"
- prepared_date: YYYY-MM-DD
- as_of: explicit Python date used for calculation

### Outputs
- item_id
- use_priority
- days_remaining
- recommended_use_date
- prepared_date
- reminder_status
- source
- reason

### Integration boundaries
Role 5 consumes use_priority rather than recreating storage rules.
Recall eligibility is handled by Role 4 and the integration pipeline.
HIGH priority does not establish food safety or recipe eligibility.

### Pending
- Confirm shared field names and demo foods.
- Verify and add selected non-meat guidance.
- Implement user-facing reminder messages.
- Package the function in the repository module.
- Test the handoff with Role 5 ranking.

### Questions for Samira
1. Can your ranking module consume use_priority, or does it currently
   require expiration_priority?
2. Which non-meat foods and storage conditions will the demo use?
3. How should UNKNOWN and PAST_REMINDER outputs be handled?

### Response to Question 1: Role 5 Ranking Compatibility

**Status: Identified integration requirement.**

Role 5 currently calculates recipe-ranking urgency using the pantry field `days_until_expiration`.

Role 3 instead returns `days_remaining`, `use_priority`, `recommended_use_date`, and `reminder_status`.

Neither `use_priority` nor `expiration_priority` is currently consumed by the Role 5 ranking function.

**Decision:** Preserve Role 3's source-backed reminder calculations and introduce an explicit mapping between Role 3 guidance and Role 5 recipe-ranking urgency.

This mapping must distinguish use-soon reminders from confirmed expiration dates and safely handle `UNKNOWN` values.

**Remaining work:** Implement and test the Role 3-to-Role 5 ranking handoff.

### Response to Question 2: Demo Foods and Storage Conditions

**Status: Proposed demo inputs; final team selection pending.**

The current Role 3 integration supports the following demonstration cases:

| Food | Storage condition | Required date | Expected result |
|---|---|---|---|
| Spinach | REFRIGERATOR | Purchase date | FoodKeeper use-soon reminder |
| Regular tomatoes | To be confirmed | N/A | UNKNOWN |
| Milk | REFRIGERATOR | Package use-by date for future support | UNKNOWN under current purchase-date evaluator |
| Cooked meat/poultry | REFRIGERATOR | Prepared date | Original Role 3 reminder |

**Spinach example:** With a purchase date of October 24, 2026, the FoodKeeper 3–7-day refrigerated reference range produces a use-soon reminder for October 27, 2026.

**Regular tomatoes:** The current FoodKeeper matching logic does not assign cherry-tomato guidance to regular tomatoes.

**Milk:** The existing FoodKeeper record refers to the package use-by date. The current implementation does not calculate a reminder from that field.

**Demo decision:** Use spinach to demonstrate newly supported FoodKeeper guidance, and use regular tomatoes to demonstrate why unsupported storage dates remain UNKNOWN.

The final demo food selection and storage inputs must be confirmed with the team.

### Response to Question 3: UNKNOWN and PAST_REMINDER Handling

**Status: Proposed integration behavior; implementation pending.**

#### UNKNOWN

`UNKNOWN` indicates that Role 3 cannot calculate a reliable use-soon reminder from the available information.

Expected behavior:

- Preserve `UNKNOWN` rather than inventing an expiration date.
- Display the explanation provided by Role 3's `reason` field.
- Assign no urgency-based recipe-ranking points without supported guidance.
- Allow the ingredient to count toward pantry availability independently of its storage priority.
- Request missing information, such as a purchase date, when appropriate.

#### PAST_REMINDER

`PAST_REMINDER` indicates that a calculated reminder date has already passed.

Expected behavior:

- Display that the recommended use-soon date has passed.
- Preserve the original date and source.
- Do not automatically label the food expired, unsafe, or recalled.
- Do not automatically prioritize the ingredient for cooking solely because its reminder is overdue.
- Require separate safety or eligibility decisions before recommending its use.

#### Role 5 Integration

Role 5 currently expects numeric `days_until_expiration` values.

Role 3 provides `days_remaining`, `use_priority`, and `reminder_status`.

An explicit integration mapping is required to handle these values safely, particularly when `days_remaining` is `None` or the reminder date has passed.

**Decision:** Treat storage reminders as prioritization guidance, not proof of food safety. Keep recipe ranking separate from food-safety and recall eligibility.

**Remaining work:** Implement and test these behaviors in the application workflow.



## Role 3 Handoff: Updated Integration Status

**Updated:** October 24, 2026

### Completed

- Preserved the original cooked-meat and poultry storage evaluator.
- Added USDA FoodKeeper data loading and validation.
- Implemented explicit FoodKeeper food matching.
- Added source-backed refrigerated spinach guidance.
- Implemented safeguards for unsupported foods and missing dates.
- Added four automated FoodKeeper tests.
- Confirmed all seven Role 3 automated tests pass.
- Documented the Role 3-to-Role 5 field mismatch.

### Remaining Integration Work

1. Confirm final demo foods and shared input fields.
2. Connect Role 3 outputs to Role 5 recipe ranking.
3. Implement user-facing reminder messages.
4. Verify UNKNOWN and PAST_REMINDER handling across the workflow.
5. Run end-to-end integration tests with the team's demo inputs.

### Integration Decision

Role 3 remains responsible for source-backed storage guidance.

Role 5 remains responsible for recipe ranking and must not
reinterpret storage reminders as verified expiration dates.

Role 4 and the integration pipeline remain responsible for
recall-related eligibility decisions.

### Handoff Status

**Role 3 FoodKeeper implementation: Unit-tested and documented.**

**Complete application integration: Pending.**

---

# Role 3: USDA FoodKeeper Integration

**Integration date:** October 24, 2026  
**Branch:** `role-3-foodkeeper-integration`  
**Module:** `modules/storage_guidance.py`  
**Tests:** `tests/test_storage.py`

## Integration Objective

Extend the original Role 3 storage guidance system to support
additional foods using the cleaned USDA FoodKeeper dataset.

The original implementation supported cooked meat and poultry
refrigeration reminders. The integration adds a separate FoodKeeper
evaluation pathway without modifying that original functionality.

## Why This Change Was Necessary

SmartPantry can identify foods through image recognition and
barcode scanning, but identifying an item does not automatically
provide reliable storage guidance.

Previously, foods such as spinach returned UNKNOWN because
the original storage rules did not cover them.

The FoodKeeper integration introduces source-backed storage
guidance while preserving UNKNOWN when the available data
does not support a calculation.

**Design principle:** We do not guess dates we cannot source.

## FoodKeeper Dataset and Source Validation

**Source:** USDA FSIS FoodKeeper Data (EN)  
**Source URL:** https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls  
**Processed dataset:** `data/processed/foodkeeper_produce_dairy_shelf_life_clean.csv`

### Dataset Preparation

The cleaned dataset contains 17 records covering selected produce and dairy products.

Each record includes:

- `food_name`: FoodKeeper food description.
- `foodkeeper_category`: Food category.
- `fridge_days_min`: Minimum refrigerated storage duration.
- `fridge_days_max`: Maximum refrigerated storage duration.
- `basis`: Reference point for interpreting the duration.
- `note`: Additional storage guidance.
- `source`: Original data source.

### Food Matching and Date Calculations

The integration uses explicit food-name matching to avoid assigning incorrect guidance.

**Spinach:** Matches the FoodKeeper entry `lettuce, leaf / spinach`, which provides a refrigerated reference range of 3–7 days from the purchase date.

**Regular tomatoes:** Remain `UNKNOWN` because the available cherry-tomato entry does not establish guidance for regular tomatoes.

**Milk:** Remains `UNKNOWN` for purchase-date calculations because its record references the package use-by date rather than a numeric duration from purchase.

### Storage Guidance Rules

A FoodKeeper-based reminder is calculated only when:

1. The food matches a supported record.
2. The storage location is `REFRIGERATOR`.
3. The record explicitly supports calculations from the purchase date.
4. A valid purchase date is available.
5. The dataset provides a valid numeric storage range.

When these requirements are satisfied, the earliest supported day is used to calculate a conservative use-soon reminder.

**Important:** FoodKeeper storage ranges are reference guidance, not guaranteed food-safety expiration dates. Unsupported or incomplete information produces `UNKNOWN` rather than an invented date.

## FoodKeeper Integration: Implementation

### 1. `load_foodkeeper_rules()`

**Purpose:** Load the cleaned FoodKeeper dataset.

- Reads the CSV using `csv.DictReader`.
- Validates that the required columns exist.
- Checks that the dataset contains records.
- Returns the records for food matching and evaluation.

**Validation result:** 17 records loaded successfully.

### 2. `find_foodkeeper_rule()`

**Purpose:** Match a recognized food to an appropriate FoodKeeper record.

The function normalizes food names and uses explicit mappings
rather than assuming that similar foods share storage guidance.

Examples:

| Recognized food | FoodKeeper match |
|---|---|
| Spinach | lettuce, leaf / spinach |
| Cherry tomatoes | cherry tomatoes |
| Regular tomato | No match |
| Milk | milk |

A successful name match does not automatically mean that a
storage date can be calculated.

### 3. `evaluate_foodkeeper()`

**Purpose:** Evaluate whether reliable storage guidance can be calculated.

The function:

1. Identifies the food using `find_foodkeeper_rule()`.
2. Verifies refrigerated storage.
3. Checks that the FoodKeeper basis supports purchase-date calculations.
4. Validates the purchase date.
5. Validates the numeric storage range.
6. Calculates the earliest recommended use-soon date.
7. Uses `evaluate_reminder()` to determine priority and reminder status.
8. Returns the guidance, source, and explanation.

When a required condition is not satisfied, the result remains
`UNKNOWN` with an explanatory reason.

### Preservation of Original Functionality

The original `evaluate_leftovers()` function remains unchanged.

FoodKeeper evaluation is implemented as a separate pathway so
that the existing cooked-meat and poultry rules continue working.

**Integration consideration:** The calling workflow must select
the appropriate evaluation pathway and supply the required fields.

## FoodKeeper Validation and Test Results

### Example: Spinach Storage Guidance

**Test input:**

- Item ID: `spinach-001`
- Food type: `Spinach`
- Storage location: `REFRIGERATOR`
- Purchase date: `2026-10-24`
- Evaluation date: `2026-10-25`

**Verified output:**

| Output field | Result |
|---|---|
| Use priority | HIGH |
| Days remaining | 2 |
| Recommended use-soon date | 2026-10-27 |
| Reminder status | UPCOMING |
| Storage range | 3–7 days |
| Storage range end date | 2026-10-31 |
| Source | USDA FSIS FoodKeeper Data (EN) |

The use-soon date is calculated using the earliest supported
storage day: October 24 + 3 days = October 27.

This is a reminder for prioritizing food use, not a guaranteed
food-safety expiration date.

### Automated Regression Testing

**Test file:** `tests/test_storage.py`

**Command:**

`python -m unittest tests.test_storage -v`

| Test | Result |
|---|---|
| test_dataset | PASS |
| test_leftovers | PASS |
| test_unsupported | PASS |
| test_foodkeeper_spinach | PASS |
| test_foodkeeper_missing_purchase_date | PASS |
| test_foodkeeper_regular_tomato | PASS |
| test_foodkeeper_milk | PASS |

**Final result: 7 tests passed.**

The original three tests continue to pass, and four additional
tests verify the FoodKeeper integration.

### Additional Manual Validation

Manual checks also confirmed:

- The cleaned FoodKeeper dataset loads 17 records.
- Spinach without a purchase date returns UNKNOWN.
- Regular tomatoes return UNKNOWN rather than borrowing cherry-tomato guidance.
- Milk returns UNKNOWN because its FoodKeeper basis does not support purchase-date calculations.

### Test Evidence

See `role3_storage_test_results.txt` in the test results folder
for the preserved original pytest baseline and the integration
test summary.

## Role 3: Integration With SmartPantry

### Role-to-Workflow Connections

Role 3 provides storage guidance after SmartPantry identifies a food item.

**Role 2: Barcode Recognition**
- Identifies packaged foods using barcode data.
- Provides available product information.
- Product names must be mapped to the food names expected by Role 3.
- A successful barcode lookup does not guarantee that storage guidance is available.

**Role 5: Image Recognition and AI Assistant**
- Identifies foods from images.
- Can provide recognized food names to the storage guidance workflow.
- Spinach is supported by the new FoodKeeper evaluation pathway.
- Regular tomatoes remain UNKNOWN unless a suitable source-backed rule is added.

**Role 3: Storage Guidance**
- Receives the food name and relevant storage information.
- Determines whether a supported storage rule exists.
- Calculates a use-soon reminder only when sufficient information is available.
- Returns UNKNOWN when the evidence does not support a calculation.

### Required Inputs

For FoodKeeper purchase-date calculations, the workflow needs:

- `item_id`
- `food_type`
- `storage_location`
- `purchase_date`

The evaluation also requires an `as_of` date.

A recognized food name alone is not sufficient to calculate a storage reminder.

### Integration Requirements

Before the complete workflow can be considered verified:

1. Confirm that Roles 2 and 5 provide compatible food-name fields.
2. Confirm how storage location and purchase date are collected.
3. Connect the appropriate storage evaluator to the application workflow.
4. Verify that supported foods produce source-backed reminders.
5. Verify that unsupported foods retain UNKNOWN.
6. Run end-to-end tests using the actual demo inputs.

### Demo Behavior

**Spinach:** Can receive a FoodKeeper-based use-soon reminder when a valid purchase date and refrigerated storage are provided.

**Regular tomatoes:** Return UNKNOWN because the current supported records do not establish an appropriate purchase-date calculation.

**Milk:** Returns UNKNOWN for purchase-date calculations because its FoodKeeper record references package use-by guidance.

**Cooked chicken:** Continues to use the original Role 3 cooked-meat and poultry rules.

### Current Status

- FoodKeeper data loading: VERIFIED
- Food matching: VERIFIED
- Spinach reminder calculation: VERIFIED
- Unsupported-food safeguards: VERIFIED
- Original Role 3 regression tests: PASS
- Role 3 automated tests: 7 PASSED
- Complete SmartPantry end-to-end integration: NOT YET VERIFIED

**Design principle:** We do not guess dates we cannot source.